# Project Analysis

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd 

import numpy as np

import os
import re

import altair as alt

from calitp_data_analysis.sql import to_snakecase
from calitp_data_analysis import calitp_color_palette as cp

from IPython.display import Markdown, HTML, display_html, display
from IPython.core.display import display

In [3]:
from IPython.display import HTML

In [4]:
pd.set_option('display.max_columns', None)

In [5]:
pd.set_option('display.max_colwidth', None)

In [6]:
GCS_PATH = "gs://calitp-analytics-data/data-analyses/project_analysis/"
GCS_PATH_AGENDAS = "gs://calitp-analytics-data/data-analyses/project_analysis/ctc_meeting_books/"

In [91]:
### read in data
full = pd.read_csv(f"{GCS_PATH}ctc_extracted_items_updated.csv")

In [92]:
display(HTML(f'<h2>Summary</h2>'))

In [8]:
 display(HTML(f"Over the last six years, there have been <strong>{len(full)}</strong> items that have appeared in CTC meeting books"))

In [9]:
(full.type.value_counts().reset_index()).rename(columns={"index":"Item Type", "program":"Number of Items"})

,Item Type,type
0,time_extension,857
1,allocation_amendment,411


In [93]:
display(HTML(f'<h3>Item Types</h3>'))

In [143]:
# ((full.groupby(['meeting_year', 'type']).size()).reset_index()).rename(columns={ 0:"Number of Items"})

In [155]:
chart = ((alt.Chart(((full.groupby(['meeting_year', 'type']).size()).reset_index().rename(columns={0:"Number of Items"})))
            .mark_bar()
            .encode(
                x=alt.X("type:O", title="Item Type"),
                 y=alt.Y('Number of Items'),
                color=alt.Color("type", scale=alt.Scale(range = cp.CALITP_CATEGORY_BRIGHT_COLORS)),
                column='meeting_year'
         )
          .properties(title="Number of Related Items in Analyses", 
                      width=100,
                      height=300)))

In [156]:
chart

alt.Chart(...)

In [145]:
display(HTML(f'<strong>Count of Item Types</strong>'))

In [146]:
(full.meeting_year.value_counts()).reset_index().rename(columns={"index":"Year", "meeting_year":"Number of Items"})

,Year,Number of Items
0,2025,570
1,2024,213
2,2020,175
3,2026,154
4,2023,87
5,2022,46
6,2021,25


In [147]:
display(HTML(f'<strong>Count of Item Types by Subtype</strong>'))

In [149]:
((full.base_code.value_counts()).reset_index()).rename(columns={"index":"Item Code", "base_code":"Number of Items"})

,Item Code,Number of Items
0,2.8a,245
1,2.5e,190
2,2.8c,160
3,2.8d,152
4,2.8v,140
5,2.8b,135
6,2.5d,75
7,2.5s,48
8,2.5b,36
9,2.8i,27


In [14]:
# alt.Chart((((full.groupby(['meeting_year', 'base_code']).size()).reset_index()).rename(columns=({0:"number_of_items"})))).mark_bar(size=70).encode(
#     x=alt.X('meeting_year', title="Meeting Year"),
#     y=alt.Y('number_of_items',),
#     color=alt.Color('base_code')
# ).properties(title="Item Types in Agendas",
#         width=700,
#         height=500)

In [15]:
district = full[full['district']<=12]

In [158]:
# ((district.groupby(['district', 'type']).size()).reset_index()).rename(columns={ 0:"Number of Items", "type":"Item Type", "district":"District"})

In [154]:
chart = ((alt.Chart((((district.groupby(['district', 'type']).size()).reset_index()).rename(columns={ 0:"Number of Items", "type":"Item Type", "district":"District"})))
            .mark_bar(size=50)
            .encode(
                x=alt.X("District", title="Caltrans District"),
                 y=alt.Y('Number of Items:Q', title="Number of Items"),
                color=alt.Color("Item Type", title="Caltrans District", scale=alt.Scale(range = cp.CALITP_CATEGORY_BRIGHT_COLORS)
                ))
          .properties(title="Number of Items in Analyses by District",
        width=800,
        height=300)))
chart

alt.Chart(...)

In [17]:
full['county'] = full['county'].str.replace(' County', '', regex=False)

In [160]:
chart = ((alt.Chart((full.county.value_counts().reset_index()))
            .mark_bar(size=8)
            .encode(
                x=alt.X("index", title="County"),
                 y=alt.Y('county', title="Number of items"),
                color=alt.Color("index", title="County", scale=alt.Scale(range = cp.CALITP_SEQUENTIAL_COLORS)
                ))
          .properties(title="Number of Items in Analyses by County",
        width=1100,
        height=300)))
chart

alt.Chart(...)

In [161]:
display(HTML(f'<strong>Count of Program Types</strong>'))

In [19]:
(full.program.value_counts().reset_index()).rename(columns={"index":"Program", "program":"Number of Items"})

,Program,Number of Items
0,SHOPP,612
1,STIP,82
2,TCEP,28
3,ATP,22
4,LPP,22
5,SCCP,1


In [53]:
# full.sample()

In [21]:
# ((full.groupby(['meeting_year', 'base_code']).size()).reset_index()).rename(columns=({0:"number_of_items"}))

In [22]:
full['prepared_division'] = full['prepared_division'].str.replace(' â€“', '', regex=False)
full['prepared_division'] = full['prepared_division'].str.replace('-', '', regex=False)
full['prepared_division'] = full['prepared_division'].str.replace(' (Acting)', '', regex=False)
full['prepared_division'] = full['prepared_division'].str.replace('  ', ' ', regex=False)

In [62]:
# full.prepared_division.value_counts()

In [63]:
display(HTML(f'<strong>Types of Items Presented by the Division of Local Assistance </strong>'))

In [164]:
((full[full['prepared_division']=="Division of Local Assistance"]).base_code.value_counts().reset_index()).rename(columns={"index":"Item Code", "base_code":"Number of Items"})

,Item Code,Number of Items
0,2.5w,14
1,2.5s,8
2,2.5c,6
3,2.8b,2
4,2.8c,1


In [61]:
display(HTML(f'<strong>Types of Items Presented by the Division of Local Assistance </strong>'))

In [32]:
(full[full['prepared_division']=="Division of Local Assistance"]).type.value_counts()

allocation_amendment    28
time_extension           3
Name: type, dtype: int64

In [58]:
display(HTML(f'<strong>Types of items in the LPP Program</strong>'))

In [57]:
(full[full['program']=="LPP"]).type.value_counts()

allocation_amendment    22
Name: type, dtype: int64

In [86]:
# full.columns

In [90]:
display(HTML(f'<h3>Allocation Amendments</h3>'))

In [47]:
full['allocation_amount'] = full['allocation_amount'].str.replace(',', '', regex=False)

In [48]:
amounts = (full[(full['type']=="allocation_amendment") & (full["allocation_amount"].notna())])

In [49]:
amounts['allocation_amount'] = amounts['allocation_amount'].astype(int)

In [72]:
# amounts.groupby('meeting_year').allocation_amount.mean().reset_index()

In [84]:
chart = ((alt.Chart((amounts.groupby('meeting_year').allocation_amount.mean().reset_index()))
            .mark_bar(size=60)
            .encode(
                x=alt.X("meeting_year", title="Meeting Year"),
                 y=alt.Y('allocation_amount', title="Allocation Amount"),
                color=alt.Color("meeting_year", title="Year", scale=alt.Scale(range = cp.CALITP_SEQUENTIAL_COLORS)
                ))
          .properties(title="Average Allocation Amendment Amount by Year",
        width=600,
        # height=300
                     )))
chart

alt.Chart(...)

In [85]:
chart = ((alt.Chart((amounts.groupby('county').allocation_amount.mean().reset_index()))
            .mark_bar(size=15)
            .encode(
                x=alt.X("county", title="County"),
                 y=alt.Y('allocation_amount', title="Allocation Amount"),
                color=alt.Color("county", title="County", scale=alt.Scale(range = cp.CALITP_SEQUENTIAL_COLORS)
                ))
          .properties(title="Average Allocation Amendment Amount by County",
        width=1000,
        height=500
                     )))
chart

alt.Chart(...)

In [87]:
extension = (full[(full['type']=="time_extension")])

In [165]:
# extension.sample()